In [1]:
# Please change the following to your own PAPER api key and secret
# or set them as environment variables (ALPACA_API_KEY, ALPACA_SECRET_KEY).
# You can get them from https://alpaca.markets/

api_key = None
secret_key = None

#### We use paper environment for this example ####
paper = True # Please do not modify this. This example is for paper trading only.
####

# Below are the variables for development this documents
# Please do not change these variables
trade_api_url = None
trade_api_wss = None
data_api_url = None
stream_data_wss = None

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

if api_key is None:
    api_key = os.environ.get('ALPACA_API_KEY')

if secret_key is None:
    secret_key = os.environ.get('ALPACA_SECRET_KEY')

In [7]:
# install alpaca-py if it is not available
try:
    import alpaca
except ImportError:
    !python3 -m pip install alpaca-py
    import alpaca

In [41]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from alpaca.trading.client import TradingClient
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.data.historical.corporate_actions import CorporateActionsClient
from alpaca.data.historical.stock import StockHistoricalDataClient
from alpaca.trading.stream import TradingStream
from alpaca.data.live.stock import StockDataStream

from alpaca.data.requests import (
    CorporateActionsRequest,
    StockBarsRequest,
    StockQuotesRequest,
    StockTradesRequest,
)
from alpaca.trading.requests import (
    ClosePositionRequest,
    GetAssetsRequest,
    GetOrdersRequest,
    LimitOrderRequest,
    MarketOrderRequest,
    StopLimitOrderRequest,
    StopLossRequest,
    StopOrderRequest,
    TakeProfitRequest,
    TrailingStopOrderRequest,
)
from alpaca.trading.enums import (
    AssetExchange,
    AssetStatus,
    OrderClass,
    OrderSide,
    OrderType,
    QueryOrderStatus,
    TimeInForce,
)
from alpaca.common.enums import (
    Sort,
)

In [9]:
# to run async code in jupyter notebook
import nest_asyncio
nest_asyncio.apply()

In [10]:
# check version of alpaca-py
alpaca.__version__

'0.43.2'

In [11]:
# setup stock historical data client
stock_historical_data_client = StockHistoricalDataClient(api_key, secret_key, url_override = data_api_url)

In [51]:
import polars as pl
import altair as alt

symbol = "SPY"

In [60]:
# get historical bars by symbol
# ref. https://docs.alpaca.markets/reference/stockbars-1
now = datetime.now(ZoneInfo("America/New_York"))
req = StockBarsRequest(
    symbol_or_symbols = [symbol],
    timeframe=TimeFrame(amount = 1, unit = TimeFrameUnit.Day), # specify timeframe
    start = now - timedelta(days = 600),                          # specify start datetime, default=the beginning of the current day.
    # end_date=None,                                        # specify end datetime, default=now
    limit = 600,                                               # specify limit
    sort = Sort.ASC,
)
hist_data = stock_historical_data_client.get_stock_bars(req)
hist_data


{   'data': {   'SPY': [   {   'close': 542.74,
                               'high': 546.95,
                               'low': 542.62,
                               'open': 544.33,
                               'symbol': 'SPY',
                               'timestamp': datetime.datetime(2024, 6, 24, 4, 0, tzinfo=TzInfo(UTC)),
                               'trade_count': 392113.0,
                               'volume': 45528654.0,
                               'vwap': 544.440534},
                           {   'close': 544.83,
                               'high': 545.2,
                               'low': 542.44,
                               'open': 543.99,
                               'symbol': 'SPY',
                               'timestamp': datetime.datetime(2024, 6, 25, 4, 0, tzinfo=TzInfo(UTC)),
                               'trade_count': 314940.0,
                               'volume': 38273346.0,
                               'vwap': 544.104149},
   

In [61]:
hist_data_polars = pl.DataFrame(hist_data.data[symbol])
hist_data_polars

symbol,timestamp,open,high,low,close,volume,trade_count,vwap
str,datetime[μs],f64,f64,f64,f64,f64,f64,f64
"""SPY""",2024-06-24 04:00:00,544.33,546.95,542.62,542.74,4.5528654e7,392113.0,544.440534
"""SPY""",2024-06-25 04:00:00,543.99,545.2,542.44,544.83,3.8273346e7,314940.0,544.104149
"""SPY""",2024-06-26 04:00:00,543.69,546.24,543.03,545.51,3.8550637e7,336264.0,544.662273
"""SPY""",2024-06-27 04:00:00,545.37,546.96,544.61,546.37,3.504148e7,318146.0,545.767008
"""SPY""",2024-06-28 04:00:00,547.16,550.28,542.95,544.22,7.5925045e7,490952.0,546.132413
…,…,…,…,…,…,…,…,…
"""SPY""",2026-02-04 05:00:00,690.35,691.45,681.76,686.19,1.052046e8,1.712102e6,686.716173
"""SPY""",2026-02-05 05:00:00,680.94,683.69,675.79,677.62,1.13610812e8,1.703564e6,679.01231
"""SPY""",2026-02-06 05:00:00,681.46,692.31,680.85,690.62,8.9127566e7,1.394903e6,687.550511


In [62]:
# Transform data
data = hist_data_polars.with_columns(# exponential moving average
                        pl.col("close").ewm_mean(span=5, adjust=False).alias('ema_5'),
                        pl.col("close").shift(1).ewm_mean(span=5, adjust=False).alias('prior_ema_5'),
                        pl.col("close").ewm_mean(span=20, adjust=False).alias('ema_20'),
                        pl.col("close").shift(1).ewm_mean(span=20, adjust=False).alias('prior_ema_20'),
                        pl.col("close").ewm_mean(span=50, adjust=True, ignore_nulls=True).alias('ema_50'),
                        pl.col("close").shift(1).ewm_mean(span=50, adjust=True, ignore_nulls=True).alias('prior_ema_50'),
                        pl.col("close").ewm_mean(span=200, adjust=True, ignore_nulls=True).alias('ema_200'),
                        pl.col("close").shift(1).ewm_mean(span=200, adjust=True, ignore_nulls=True).alias('prior_ema_200'),
                        # Simple moving avaerage
                        pl.col("close").rolling_mean(10).alias("close_MA10"),
                        pl.col("close").rolling_mean(20).alias("close_MA20"))
data = data.with_columns(pl.when((pl.col("ema_5") > pl.col("ema_20")) & (pl.col("prior_ema_5") <= pl.col("prior_ema_20")))
                            .then(pl.col("close")*0.98).alias("sig_bull_5_20"),
                        pl.when((pl.col("ema_5") < pl.col("ema_20")) & (pl.col("prior_ema_5") >= pl.col("prior_ema_20")))
                            .then(pl.col("close")*0.98).alias("sig_bear_5_20"),
                        pl.when((pl.col("ema_20") > pl.col("ema_50")) & (pl.col("prior_ema_20") <= pl.col("prior_ema_50")))
                            .then(pl.col("close")*0.98).alias("sig_bull_20_50"),
                        pl.when((pl.col("ema_20") < pl.col("ema_50")) & (pl.col("prior_ema_20") >= pl.col("prior_ema_50")))
                            .then(pl.col("close")*0.98).alias("sig_bear_20_50"),
                        pl.when((pl.col("ema_50") > pl.col("ema_200")) & (pl.col("prior_ema_50") <= pl.col("prior_ema_200")))
                            .then(pl.col("close")*0.98).alias("sig_bull_50_200"),
                        pl.when((pl.col("ema_50") < pl.col("ema_200")) & (pl.col("prior_ema_50") >= pl.col("prior_ema_200")))
                            .then(pl.col("close")*0.98).alias("sig_bear_50_200"),)
data

symbol,timestamp,open,high,low,close,volume,trade_count,vwap,ema_5,prior_ema_5,ema_20,prior_ema_20,ema_50,prior_ema_50,ema_200,prior_ema_200,close_MA10,close_MA20,sig_bull_5_20,sig_bear_5_20,sig_bull_20_50,sig_bear_20_50,sig_bull_50_200,sig_bear_50_200
str,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""SPY""",2024-06-24 04:00:00,544.33,546.95,542.62,542.74,4.5528654e7,392113.0,544.440534,542.74,null,542.74,null,542.74,null,542.74,null,null,null,null,null,null,null,null,null
"""SPY""",2024-06-25 04:00:00,543.99,545.2,542.44,544.83,3.8273346e7,314940.0,544.104149,543.436667,542.74,542.939048,542.74,543.8059,542.74,543.790225,542.74,null,null,533.9334,null,null,533.9334,533.9334,null
"""SPY""",2024-06-26 04:00:00,543.69,546.24,543.03,545.51,3.8550637e7,336264.0,544.662273,544.127778,543.436667,543.1839,542.939048,544.396803,543.8059,544.369225,543.790225,null,null,null,null,null,null,null,null
"""SPY""",2024-06-27 04:00:00,545.37,546.96,544.61,546.37,3.504148e7,318146.0,545.767008,544.875185,544.127778,543.487338,543.1839,544.920087,544.396803,544.876947,544.369225,null,null,null,null,null,null,null,null
"""SPY""",2024-06-28 04:00:00,547.16,550.28,542.95,544.22,7.5925045e7,490952.0,546.132413,544.65679,544.875185,543.557116,543.487338,544.768649,544.920087,544.742917,544.876947,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SPY""",2026-02-04 05:00:00,690.35,691.45,681.76,686.19,1.052046e8,1.712102e6,686.716173,690.260661,692.295991,690.089617,690.500103,685.217093,685.177382,652.749715,652.407673,691.899,690.916,null,null,null,null,null,null
"""SPY""",2026-02-05 05:00:00,680.94,683.69,675.79,677.62,1.13610812e8,1.703564e6,679.01231,686.047107,690.260661,688.902035,690.089617,684.919167,685.217093,653.00148,652.749715,690.763,690.318,null,664.0676,null,null,null,null
"""SPY""",2026-02-06 05:00:00,681.46,692.31,680.85,690.62,8.9127566e7,1.394903e6,687.550511,687.571405,686.047107,689.06565,688.902035,685.142729,684.919167,653.382231,653.00148,690.902,690.3735,null,null,null,null,null,null


In [63]:
# Plot data using a long dataframe format for price data

# First convert from wide to long using unpivot
columns = ['timestamp', 'close', 'ema_50', 'ema_200']
line_columns = data.select(columns)
long_data = line_columns.unpivot(index='timestamp', on=['close', 'ema_50', 'ema_200'], variable_name='type', value_name='value')

line_chart = (
    alt.Chart(long_data)
    .mark_line(strokeWidth=1)
    .encode(x=alt.X("timestamp:T",
                    title='',
                    axis=alt.Axis(labelAngle=-45, tickColor="#09080B", domainColor="#09080B", grid=False, labelFontSize=10, titleFontSize=12)),
            y=alt.Y("value:Q",
                    title='Close',
                    scale=alt.Scale(zero=False),
                    axis=alt.Axis(format='$.2f', tickColor="#09080B", domainColor="#09080B", gridColor="#9DCDF8", grid=True, labelFontSize=10, titleFontSize=12)
                    ),
            color="type:N",
            tooltip=['timestamp:T', 'value:Q'])
    ).properties(title={'text': 'Stock Price Analysis',
                        'subtitle': ['50 and 200-day exponential moving average overlay'],
                        'fontSize': 16
                },
                width=500,
                height=300,
    )

xrule = (
        alt.Chart()
        .mark_rule(color="#00FFFF", strokeWidth=2)
        .encode(x=alt.datum(alt.DateTime(year=2024, month="November")))
)

yrule = (
    alt.Chart().mark_rule(color="#1010AA", strokeDash=[12, 6], size=2).encode(y=alt.datum(550))
)

bull_sig = (
    alt.Chart(data)
    .mark_point(shape='triangle-up',color="#0F9A0F",size=30, strokeWidth=3)
    .encode(x='timestamp:T',
            y='sig_bull_50_200'))

bear_sig = (
    alt.Chart(data)
    .mark_point(shape='triangle-down',color='#FF0000',size=30, strokeWidth=3)
    .encode(x='timestamp:T',
            y='sig_bear_50_200'))

#Layer the above five together
line_chart = (line_chart + xrule + yrule + bull_sig + bear_sig).interactive()

bar_chart = alt.Chart(data).mark_bar(color="#4286B4",opacity=1.0
    ).encode(x=alt.X("timestamp:T",
                title='Date',
                axis=alt.Axis(labelAngle=-45, tickColor="#09080B", domainColor="#09080B", grid=False, labelFontSize=10, titleFontSize=12)
                ),
            y=alt.Y('volume:Q',
                    title='Volume',
                    axis=alt.Axis(tickColor="#09080B", domainColor="#09080B", gridColor="#9DCDF8", grid=True, labelFontSize=10,titleFontSize=12)
                    ),
            tooltip=['timestamp:T', 'Volume:Q'],
    ).properties(title='Volume',
                width=500,
                height=150,)

# Vertically combine the charts
combined_chart = alt.vconcat(line_chart, bar_chart).resolve_scale(x='shared').configure(background="#CBE1F6")
combined_chart.show()

alt.VConcatChart(...)